In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("pertemuan7") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi spark: ", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 01:04:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi spark:  3.5.9


In [2]:
np.random.seed(77)

# SUMBER 1: Data toko/cabang(JSON) - tabel referensi kecil
toko_list = ["Toko Magelang", "Toko Yogyakarta", "Toko Semarang", "Toko Solo"]
data_toko = [
    {"toko_id": i + 1, "nama_toko": t, "kota": t.split()[1], "manager": np.random.choice(["Rani", "Joko", "Sari", "Bayu"])}
    for i, t in enumerate(toko_list)
]
with open("data_toko.json", "w") as f:
    for t in data_toko:
        f.write(json.dumps(t) + "\n")

# SUMBER 2: Data produk (CSV) - tabel referensi kecil
data_produk = [
    {"produk_id": i, "nama_produk": f"Produk-{i}",
     "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan"]),
     "harga": int(np.random.choice([50000, 100000, 200000]))}
    for i in range(1, 21)
]
pd.DataFrame(data_produk).to_csv("data_produk.csv", index=False)

# SUMBER 3: Data transaksi triwulan (CSV) - data utama, jauh lebih besar
n = 5000
data_transaksi = pd.DataFrame({
    "trx_id": [f"T{i}" for i in range(n)],
    "toko_id": np.random.randint(1, 5, size=n),
    "product_id": np.random.randint(1, 21, size=n),
    "qty": np.random.randint(1, 6, size=n),
})
data_transaksi.to_csv("data_transaksi_triwulan.csv", index=False)

print("Ketiga sumber data berhasil dibuat:")
print(f"- data_toko.json {len(data_toko)} baris")
print(f"- data_produk.csv {len(data_produk)} baris")
print(f"- data_transaksi_triwulan.csv {len(data_transaksi)} baris")

Ketiga sumber data berhasil dibuat:
- data_toko.json 4 baris
- data_produk.csv 20 baris
- data_transaksi_triwulan.csv 5000 baris


In [8]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_produk = df_produk.withColumnRenamed("produk_id", "product_id")

df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


In [9]:
# 1 

df_gabungan.createOrReplaceTempView("gabungan")

spark.sql('''
    SELECT kategori,
        SUM(qty) AS total_qty
    FROM gabungan
    GROUP BY kategori
    ORDER BY total_qty DESC
    LIMIT 1
''').show()

+----------+---------+
|  kategori|total_qty|
+----------+---------+
|Elektronik|     6736|
+----------+---------+



In [11]:
# 2

#1. tOTAL PENJUALAN BERNILAI RUPIAH TIAP TOKO DI TIAP KATEGORI
penjualan_toko_kategori = (
    df_gabungan
    .groupBy("kategori", "nama_toko")
    .agg(spark_sum("total_penjualan").alias("total_penjualan"))
)

#2. WINDOW: PERANGKAT TOKO DI DALAM KATEGORI
window_spec = Window.partitionBy("kategori").orderBy(col("total_penjualan").desc())

toko_terbaik = (
    penjualan_toko_kategori
    .withColumn("peringkat", row_number().over(window_spec))
    .filter(col("peringkat") == 1)
)

toko_terbaik.show()

+----------+-------------+---------------+---------+
|  kategori|    nama_toko|total_penjualan|peringkat|
+----------+-------------+---------------+---------+
|Elektronik|Toko Magelang|      202150000|        1|
|   Fashion|Toko Semarang|      123000000|        1|
|   Makanan|Toko Magelang|      108200000|        1|
+----------+-------------+---------------+---------+



In [13]:
# Buat ringkasan eksekutif
ringkasan_eksekutif = (
     df_gabungan
    .groupBy("nama_toko", "kota", "manager")
    .agg(
        spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
        count("trx_id").alias("jumlah_transaksi"),
        avg("total_penjualan").alias("rata_rata_nilai_transaksi")
    )
    .orderBy(col("total_penjualan_triwulan").desc())
)


!hdfs dfs -mkdir -p /user/azka/pertemuan7/processed

df_gabungan.write.mode("overwrite").partitionBy()